# Montha 2025 Hero Figure

Phase 1 of Resolve — run top-to-bottom in Colab or Kaggle.

**Label:** replay

**Output:** `outputs/montha/hero.png`, `hero.svg`, `summary.json`

**Reproducibility:** This notebook and `python -m resolve hero --config configs/montha.yaml` produce identical outputs.

In [ ]:
# Install the package and dependencies
!pip install -e . -q
# Set up Earthdata credentials (fill in or use ~/.netrc)
import os
# os.environ['EARTHDATA_USERNAME'] = 'your_username'
# os.environ['EARTHDATA_PASSWORD'] = 'your_password'

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s'
)

from resolve.hero import run_hero

# Full pipeline:
#   Step 1: Ingest IFS ENS runs to Zarr cache
#   Step 2: Fetch IMD + IMERG truth, run alignment check
#   Step 3: Select target day (largest area >= 115.6 mm in Oct 26-31)
#   Step 4: Accumulate 24 h member rain per init
#   Step 5: Compute exceedance masks + probabilities
#   Step 6: 8-conn object detection
#   Step 7: Hungarian matching across runs, drift/IoU, lock-on
#   Step 8: FSS table + earned width per lead
#   Step 9: 3-panel hero figure (PNG 300 dpi + SVG)
#   Step 10: summary.json + provenance sidecars
run_hero('configs/montha.yaml')

In [ ]:
from IPython.display import Image
Image('outputs/montha/hero.png')

In [ ]:
import json
summary = json.load(open('outputs/montha/summary.json'))
print('Label:', summary['label'])
print('Target date:', summary['target_date'])
print('Lock-on run:', summary['lockon_run'])
print()
print('Earned width per lead:')
for lead, val in summary['earned_width_per_lead'].items():
    km = val['earned_width_km']
    cells = val['earned_width_cells']
    if km:
        print(f'  {lead}: {cells} cells ({km:.0f} km)')
    else:
        print(f'  {lead}: no skill')
print()
print('Alignment check:', summary['alignment'])

In [ ]:
# FSS table
import pandas as pd
fss = summary['fss_table']
df = pd.DataFrame(fss).T
df.index.name = 'Lead'
df.columns = [f'{c} cells' for c in df.columns]
print(df.to_string())